# SINCA: Calidad del Aire y Meteorología

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Observatory/datopia-notebooks/blob/main/medio-fisico/sinca/demo.ipynb)
[![Licencia](https://img.shields.io/badge/licencia-MIT-blue.svg)](../../LICENSE)
[![Dataset](https://img.shields.io/badge/dataset-SINCA-green.svg)]()
[![Actualización](https://img.shields.io/badge/actualización-diaria-brightgreen.svg)]()
[![Python](https://img.shields.io/badge/python-3.10%2B-blue.svg)]()

---

## Descripción

Acceso al dataset de calidad del aire y meteorología de **SINCA** (Sistema de Información Nacional
de Calidad del Aire, Ministerio del Medio Ambiente de Chile) a través del **Datopia Lakehouse**.
212 estaciones activas en las 16 regiones de Chile, cobertura desde 1971 (red completa desde 2000).

El acceso es directo vía S3 con credenciales temporales, sin descarga de archivos.

### Contenido

1. Configuración y autenticación
2. Exploración del dataset (catálogo, esquema, estaciones, variables medidas, calidad)
3. Mapa de estaciones: PM2.5 promedio últimos 30 días
4. Series de tiempo: PM2.5 en tres estaciones representativas (5 años)
5. Series de tiempo: múltiples variables en una estación (5 años)
6. Análisis exploratorio mínimo (5 años)
7. Modelo lineal simple: tendencia de PM2.5 y PM10 en La Florida (10 años)

### Requisitos

Cuenta en el Datopia Lakehouse · Python 3.10+ · Las dependencias se instalan automáticamente

In [ ]:
# @title Instalación de dependencias
import importlib, subprocess, sys

for paquete in ["requests", "duckdb", "pandas", "matplotlib", "numpy", "ipyleaflet", "ipywidgets"]:
    if importlib.util.find_spec(paquete) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

import json, os, getpass, pathlib
import requests, duckdb
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipyleaflet
from ipywidgets import HTML as PopupHTML

EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

print("Dependencias listas.")

In [ ]:
# @title Configuración de credenciales
print(f"Entorno: {'Google Colab' if EN_COLAB else 'local'}")

URL_API = "https://ee98cnfz7e.execute-api.us-west-2.amazonaws.com/prod"

if EN_COLAB:
    EMAIL = input("Email: ").strip()
    PASSWORD = getpass.getpass("Contraseña: ")
else:
    ruta_cfg = pathlib.Path("../../config.json")
    if not ruta_cfg.exists():
        raise FileNotFoundError(
            f"Archivo de configuración no encontrado en {ruta_cfg.resolve()}.\n"
            "Copia config.example.json → config.json y completa tus credenciales."
        )
    cfg = json.loads(ruta_cfg.read_text())
    EMAIL = cfg.get("test_user", {}).get("email") or input("Email: ").strip()
    PASSWORD = cfg.get("test_user", {}).get("password") or getpass.getpass("Contraseña: ")

print(f"API    : {URL_API}")
print(f"Usuario: {EMAIL}")

---
## 1. Autenticación y conexión a S3

In [ ]:
# Iniciar sesión y obtener credenciales temporales S3
resp_login = requests.post(
    f"{URL_API}/auth/login",
    json={"email": EMAIL, "password": PASSWORD},
    timeout=30,
)
resp_login.raise_for_status()
token = resp_login.json()["id_token"]

resp_s3 = requests.post(
    f"{URL_API}/auth/session/s3",
    headers={"Authorization": f"Bearer {token}"},
    timeout=30,
)
resp_s3.raise_for_status()
creds = resp_s3.json()

print("Sesión iniciada")
print(f"  Bucket : {creds['bucket']}")
print(f"  Región : {creds['region']}")
print(f"  Expira : {creds['expires_at']}")

In [ ]:
# Configurar DuckDB con credenciales S3
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs")
con.execute(f"""
    CREATE OR REPLACE SECRET s3 (
        TYPE          S3,
        KEY_ID        '{creds["access_key_id"]}',
        SECRET        '{creds["secret_access_key"]}',
        SESSION_TOKEN '{creds["session_token"]}',
        REGION        '{creds["region"]}'
    )
""")

BASE = f"s3://{creds['bucket']}/categoria=medio-fisico/pais=cl/fuente=sinca"
print("DuckDB conectado a S3.")

---
## 2. Exploración del dataset

In [ ]:
# Metadatos del dataset de mediciones (dataset.json en la raíz del path S3)
meta = (
    con.execute(f"SELECT * FROM read_json('{BASE}/tipo=medicion-diaria/version=v1/dataset.json')")
    .df()
    .iloc[0]
    .to_dict()
)

print(meta["description"][:600] + "...")
print()
print(f"Formato      : {meta['format']}")
particiones = list(meta["partition_keys"])
print(f"Particiones  : {particiones if particiones else '(ninguna -- layout plano, ver celda siguiente)'}")

cols = pd.DataFrame([dict(c) for c in meta["columns"]])[["name", "type", "comment"]]
cols

In [ ]:
# Esquema real. Layout plano (2026-08-09): sin particiones year=/month=/day=,
# solo un puñado de archivos part-NNNNN.parquet -- ver manifest.json para la lista.
# estacion_fk/variable_fk/periodicidad_fk son claves surrogate (int), no strings --
# hay que hacer join contra tipo=estaciones/tipo=variables para obtener
# estacion_id/variable_code legibles (ver celdas siguientes).
con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{BASE}/tipo=medicion-diaria/version=v1/part-*.parquet')
    LIMIT 0
""").df()[["column_name", "column_type", "null"]]

In [ ]:
# Catálogo de estaciones activas (tipo=estaciones, SCD-2, fecha_fin IS NULL = activa)
# estacion_fk incluido: es la clave que usa medicion-diaria para referenciar la estación.
estaciones = con.execute(f"""
    SELECT estacion_fk, estacion_id, nombre, region, comuna, lon, lat, inicio_operacion
    FROM read_parquet('{BASE}/tipo=estaciones/version=v1/part-00000.parquet')
    WHERE fecha_fin IS NULL
""").df()

print(f"Estaciones activas: {len(estaciones)}")
print(f"Sin coordenadas (geometry nula en origen): {estaciones['lon'].isna().sum()}")

resumen_regiones = estaciones.groupby("region").agg(
    estaciones=("estacion_id", "count"),
    estacion_mas_antigua=("inicio_operacion", "min"),
    estacion_mas_reciente=("inicio_operacion", "max"),
).sort_values("estaciones", ascending=False)
resumen_regiones

In [ ]:
# Tablas de referencia (tipo=variables, tipo=calidad): diccionarios estáticos
variables_df = con.execute(f"""
    SELECT * FROM read_parquet('{BASE}/tipo=variables/version=v1/part-00000.parquet')
""").df()
calidad_df = con.execute(f"""
    SELECT * FROM read_parquet('{BASE}/tipo=calidad/version=v1/part-00000.parquet')
""").df()

print(f"Variables documentadas: {len(variables_df)}")
print(calidad_df.to_string(index=False))
variables_df.head(8)

**`medicion-diaria` solo tiene números (2026-08-09).** `estacion_fk`/`variable_fk`/
`periodicidad_fk` son claves surrogate enteras, no los códigos de texto (`estacion_id`,
`variable_code`, `"diario"`/`"horario"`) -- guardar el texto directamente inflaba ~48x el
tamaño en memoria al procesar el archivo completo (17.3GB para 115M filas, vs 360MB
comprimido), muy por sobre lo que el pipeline diario puede manejar. Para obtener los códigos
legibles hay que hacer `JOIN` contra `tipo=estaciones`/`tipo=variables` (por `estacion_fk`/
`variable_fk`) -- exactamente lo que hacen las consultas de este notebook de aquí en
adelante. `periodicidad_fk`: `0=horario, 1=diario, 2=discreto` (sin tabla de referencia --
son solo 3 valores fijos).

In [ ]:
# Registrar estaciones/variables como vistas DuckDB -- permite hacer JOIN directo en
# SQL contra medicion-diaria sin releer los parquet chicos en cada consulta.
con.register("estaciones_df", estaciones)
con.register("variables_df", variables_df)
print("Vistas registradas: estaciones_df, variables_df")

In [ ]:
# tipo=estaciones-variables: qué variables mide cada estación, con nombre desde tipo=variables
# NOTA (2026-08-05): la columna se llama "variable_tipo" en ambas tablas, no "tipo" --
# ver Notas de actualización 2026-08-05 en el README.
variables_florida = con.execute(f"""
    SELECT ev.variable_code, v.variable_nombre, ev.variable_tipo, v.unidad, ev.periodicidad
    FROM read_parquet('{BASE}/tipo=estaciones-variables/version=v1/part-00000.parquet') ev
    LEFT JOIN read_parquet('{BASE}/tipo=variables/version=v1/part-00000.parquet') v
      ON ev.variable_code = v.variable_code
    WHERE ev.estacion_id = 'D12'
    ORDER BY ev.variable_code
""").df()
print(f"La Florida (D12) mide {len(variables_florida)} combinaciones variable/periodicidad:")
variables_florida

**Variables meteorológicas (2026-08-05): 6 de 7 ya tienen mediciones.** Hasta esa fecha,
`medicion-diaria` no tenía ninguna fila para las 7 variables meteorológicas pese a estar
correctamente catalogadas en `tipo=variables`/`tipo=estaciones-variables` — el extractor nunca
llegaba a pedirlas al CGI de SINCA por un bug de discovery. Ya corregido: TEMP, RHUM, WSPD, PRES,
RAIN y GLOB tienen datos reales (siempre `periodicidad = 'horario'`, SINCA no publica agregado
diario para meteo). **WDIR (dirección del viento) sigue sin datos** — el CGI de SINCA devuelve
una tabla de "rosa de los vientos" en vez de una serie de tiempo para esa variable específica;
limitación de la fuente, no de nuestro scraper.

In [ ]:
# Temperatura horaria en La Florida (D12) -- variable meteorológica, antes vacía
temp_d12 = con.execute(f"""
    SELECT m.fecha, m.valor AS temperatura_c
    FROM read_parquet('{BASE}/tipo=medicion-diaria/version=v1/part-*.parquet') m
    JOIN estaciones_df e ON m.estacion_fk = e.estacion_fk
    JOIN variables_df v ON m.variable_fk = v.variable_fk
    WHERE e.estacion_id = 'D12'
      AND v.variable_code = 'TEMP' AND m.periodicidad_fk = 0  -- 0 = horario
      AND m.fecha >= TIMESTAMP '2026-07-01' AND m.fecha < TIMESTAMP '2026-07-16'
    ORDER BY m.fecha
""").df()

print(f"Filas TEMP cargadas: {len(temp_d12)}")
temp_d12.head()

---
## 3. Mapa de estaciones

In [ ]:
# PM2.5 promedio de los últimos 30 días por estación
pm25_estaciones = con.execute(f"""
    SELECT e.estacion_id, AVG(m.valor) AS pm25_promedio, COUNT(*) AS n_dias
    FROM read_parquet('{BASE}/tipo=medicion-diaria/version=v1/part-*.parquet') m
    JOIN estaciones_df e ON m.estacion_fk = e.estacion_fk
    JOIN variables_df v ON m.variable_fk = v.variable_fk
    WHERE v.variable_code = 'PM25' AND m.periodicidad_fk = 1  -- 1 = diario
      AND m.fecha >= current_date - INTERVAL 30 DAY
    GROUP BY e.estacion_id
""").df()

mapa_df = pm25_estaciones.merge(estaciones, on="estacion_id").dropna(subset=["lon", "lat"])
print(f"Estaciones con dato de PM2.5 en el mapa: {len(mapa_df)} / {len(estaciones)} activas")

norm = mcolors.Normalize(vmin=mapa_df["pm25_promedio"].min(), vmax=mapa_df["pm25_promedio"].max())
cmap = mpl.colormaps["YlOrRd"]

mapa = ipyleaflet.Map(center=(-33.5, -70.9), zoom=4, scroll_wheel_zoom=True)
for _, row in mapa_df.iterrows():
    color = mcolors.to_hex(cmap(norm(row["pm25_promedio"])))
    marcador = ipyleaflet.CircleMarker(
        location=(row["lat"], row["lon"]),
        radius=7, color=color, fill_color=color, fill_opacity=0.85, weight=1,
    )
    marcador.popup = PopupHTML(
        value=(
            f"<b>{row['nombre']}</b><br>{row['region']}<br>"
            f"PM2.5 promedio: {row['pm25_promedio']:.1f} µg/m³ ({row['n_dias']} días)"
        )
    )
    mapa.add(marcador)

mapa

---
## 4. Series de tiempo: PM2.5 en estaciones representativas (5 años)

Tres estaciones activas con cobertura reciente, elegidas por dispersión geográfica:
La Florida (Santiago, Región Metropolitana), Bomberos (Antofagasta, minería del norte) y
Osorno (Región de los Lagos, sur).

In [ ]:
ESTACIONES_TS = ["D12", "230", "A01"]  # La Florida, Bomberos (Antofagasta), Osorno

pm25_series = con.execute(f"""
    SELECT e.estacion_id, CAST(m.fecha AS DATE) AS fecha, m.valor
    FROM read_parquet('{BASE}/tipo=medicion-diaria/version=v1/part-*.parquet') m
    JOIN estaciones_df e ON m.estacion_fk = e.estacion_fk
    JOIN variables_df v ON m.variable_fk = v.variable_fk
    WHERE e.estacion_id IN ({", ".join(f"'{e}'" for e in ESTACIONES_TS)})
      AND v.variable_code = 'PM25' AND m.periodicidad_fk = 1  -- 1 = diario
      AND m.fecha >= DATE '2021-07-01' AND m.fecha < DATE '2026-07-01'
    ORDER BY e.estacion_id, fecha
""").df()

nombres = estaciones.set_index("estacion_id")["nombre"].to_dict()
print(f"Filas cargadas: {len(pm25_series):,}")

COLORES = {"D12": "#1565C0", "230": "#E65100", "A01": "#2E7D32"}
fig, ax = plt.subplots(figsize=(11, 4.5))
for est_id in ESTACIONES_TS:
    sub = pm25_series[pm25_series["estacion_id"] == est_id]
    ax.plot(sub["fecha"], sub["valor"], label=nombres[est_id], color=COLORES[est_id], linewidth=1.0)

ax.set_title("PM2.5 diario (µg/m³): jul 2021 a jun 2026", fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Series de tiempo: múltiples variables en una estación (5 años)

Perfil de contaminantes en La Florida (Santiago): PM2.5, PM10 y O3, mismo período de 5 años
que la sección anterior.

In [ ]:
CODIGOS_PERFIL = ["PM25", "PM10", "O3"]
nombres_var = variables_df.set_index("variable_code")["variable_nombre"].to_dict()
VARIABLES_PERFIL = {c: nombres_var.get(c, c) for c in CODIGOS_PERFIL}

perfil = con.execute(f"""
    SELECT CAST(m.fecha AS DATE) AS fecha, v.variable_code, m.valor
    FROM read_parquet('{BASE}/tipo=medicion-diaria/version=v1/part-*.parquet') m
    JOIN estaciones_df e ON m.estacion_fk = e.estacion_fk
    JOIN variables_df v ON m.variable_fk = v.variable_fk
    WHERE e.estacion_id = 'D12'
      AND v.variable_code IN ({", ".join(f"'{v}'" for v in CODIGOS_PERFIL)})
      AND m.periodicidad_fk = 1  -- 1 = diario
      AND m.fecha >= DATE '2021-07-01' AND m.fecha < DATE '2026-07-01'
    ORDER BY v.variable_code, fecha
""").df()
perfil["variable"] = perfil["variable_code"].map(VARIABLES_PERFIL)

fig_perfil, ax = plt.subplots(figsize=(11, 4.5))
for code, color in zip(CODIGOS_PERFIL, ["#C62828", "#6A1B9A", "#00838F"]):
    var = VARIABLES_PERFIL[code]
    sub = perfil[perfil["variable"] == var]
    ax.plot(sub["fecha"], sub["valor"], label=var, color=color, linewidth=1.0)

ax.set_title(f"La Florida (Santiago): {', '.join(VARIABLES_PERFIL.values())} diario, 2021-2026", fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("µg/m³")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Análisis exploratorio mínimo

In [ ]:
# Estadísticas descriptivas: PM2.5 en La Florida, mismo período (2021-2026)
pm25_d12 = perfil[perfil["variable"] == VARIABLES_PERFIL["PM25"]]["valor"]
print("PM2.5 La Florida (jul 2021 - jun 2026):")
print(pm25_d12.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pm25_d12, bins=30, color="#455A64", edgecolor="white")
axes[0].set_title("Distribución PM2.5 diario")
axes[0].set_xlabel("µg/m³")
axes[0].set_ylabel("Días")

calidad = con.execute(f"""
    SELECT m.calidad_id, COUNT(*) AS n
    FROM read_parquet('{BASE}/tipo=medicion-diaria/version=v1/part-*.parquet') m
    JOIN variables_df v ON m.variable_fk = v.variable_fk
    WHERE v.variable_code = 'PM25' AND m.periodicidad_fk = 1  -- 1 = diario
      AND m.fecha >= DATE '2021-07-01' AND m.fecha < DATE '2026-07-01'
    GROUP BY m.calidad_id ORDER BY m.calidad_id
""").df()
calidad = calidad.merge(calidad_df, on="calidad_id", how="left")
axes[1].bar(calidad["descripcion"], calidad["n"], color="#00695C")
axes[1].set_title("Filas PM2.5 2021-2026 por calidad_id")
axes[1].set_ylabel("Filas")

plt.tight_layout()
plt.show()

---
## 7. Modelo lineal simple: tendencia de PM2.5 y PM10 en La Florida (10 años)

Ajuste de una tendencia lineal simple sobre el promedio diario de PM2.5 y PM10 en La Florida
(Santiago), cada variable por separado, 2017-2026. Solo para demostrar capacidad de análisis,
no un modelo predictivo serio (sin estacionalidad, sin validación cruzada).

In [ ]:
AÑO_INICIO_TENDENCIA = 2017

tendencia_pm = con.execute(f"""
    SELECT CAST(m.fecha AS DATE) AS fecha, v.variable_code, m.valor
    FROM read_parquet('{BASE}/tipo=medicion-diaria/version=v1/part-*.parquet') m
    JOIN estaciones_df e ON m.estacion_fk = e.estacion_fk
    JOIN variables_df v ON m.variable_fk = v.variable_fk
    WHERE e.estacion_id = 'D12'
      AND v.variable_code IN ('PM25', 'PM10')
      AND m.periodicidad_fk = 1  -- 1 = diario
      AND m.fecha >= DATE '{AÑO_INICIO_TENDENCIA}-01-01'
    ORDER BY v.variable_code, fecha
""").df()

print(f"Días con dato: {len(tendencia_pm):,} ({tendencia_pm['fecha'].min()} a {tendencia_pm['fecha'].max()})")

COLORES_TENDENCIA = {"PM25": ("#90A4AE", "#1565C0"), "PM10": ("#BCAAA4", "#D84315")}
fig, ax = plt.subplots(figsize=(11, 4.5))
for codigo in ["PM25", "PM10"]:
    sub = tendencia_pm[tendencia_pm["variable_code"] == codigo].dropna(subset=["valor"])
    x = (sub["fecha"] - sub["fecha"].min()).dt.days.to_numpy()
    y = sub["valor"].to_numpy()

    pendiente, intercepto = np.polyfit(x, y, 1)
    y_ajustado = pendiente * x + intercepto
    ss_res = np.sum((y - y_ajustado) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    nombre = nombres_var.get(codigo, codigo)
    print(f"{nombre:6s} pendiente: {pendiente:+.5f} µg/m³/día ({pendiente * 365:+.3f} µg/m³/año), R²: {r2:.4f}")

    color_pts, color_linea = COLORES_TENDENCIA[codigo]
    ax.scatter(sub["fecha"], y, s=3, alpha=0.25, color=color_pts, label=f"{nombre} diario")
    ax.plot(sub["fecha"], y_ajustado, color=color_linea, linewidth=2, label=f"Tendencia {nombre}")

ax.set_title("La Florida (Santiago): PM2.5 y PM10 diario, tendencia lineal 2017-2026", fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("µg/m³")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

*Datos provistos por [Data Observatory](https://dataobservatory.net) · Fuente: SINCA, Ministerio del Medio Ambiente de Chile*